# Mistral-7B Abstract Evaluator (Modular)

This notebook orchestrates reusable utilities from:
- `experiments/utils` (model-agnostic)
- `experiments/mistral/utils` (Mistral-specific)

Model used in this notebook:
- `mistralai/Mistral-7B-Instruct-v0.3`

Constraints baked in:
- No quantization (`use_4bit=False`)
- L4 24GB-friendly train/eval batch sizes
- No resume-from-epoch-2 flow


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd


def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "experiments").exists() and (p / "data").exists():
            return p
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


In [ ]:
from experiments.mistral.utils.chat import add_messages_and_targets
from experiments.mistral.utils.configs import build_data_paths, build_mistral7_default_config
from experiments.mistral.utils.modeling import load_mistral_model_for_inference
from experiments.mistral.utils.pipeline import (
    estimate_mistral_token_percentiles,
    evaluate_base_model,
    evaluate_saved_adapters,
    evaluate_single_adapter,
)
from experiments.mistral.utils.training import train_mistral7

from experiments.utils.data import (
    clean_train_val_test,
    load_train_val_test_dfs,
    score_distribution,
)
from experiments.utils.datasets_io import export_split_jsonl, to_hf_dataset_dict
from experiments.utils.evaluation import parse_rationale, parse_score
from experiments.utils.generation import generate_predictions_from_messages
from experiments.utils.logging_utils import setup_logger
from experiments.utils.runtime import configure_wandb_dir, set_global_seed


In [ ]:
cfg = build_mistral7_default_config(PROJECT_ROOT)

# --- Core run identity ---
cfg.model_name = "mistralai/Mistral-7B-Instruct-v0.3"
cfg.run_name = "mistral7b_abstract_evaluator_lora_l4_modular"

# --- Data paths (edit if you want different splits) ---
cfg.data_paths = build_data_paths(
    train_path=PROJECT_ROOT / "data/data/train/all.jsonl",
    val_path=PROJECT_ROOT / "data/data/val/all.jsonl",
    test_path=PROJECT_ROOT / "data/data/test/all.jsonl",
)

# --- Trainer config (L4 24GB-safe + no quantization) ---
cfg.max_seq_length = 2048
cfg.use_4bit = False
cfg.train.num_train_epochs = 30
cfg.train.per_device_train_batch_size = 2
cfg.train.per_device_eval_batch_size = 4
cfg.train.gradient_accumulation_steps = 4
cfg.train.learning_rate = 8e-5
cfg.train.logging_steps = 5

# Eval/save once per epoch
cfg.train.eval_strategy = "epoch"
cfg.train.eval_steps = None
cfg.train.save_strategy = "epoch"
cfg.train.save_steps = None
cfg.train.save_total_limit = 20

# Stop when validation loss no longer improves
cfg.train.early_stopping_patience = 3
cfg.train.early_stopping_threshold = 0.0

# Throughput/stability settings
cfg.train.dataloader_num_workers = 4
cfg.train.dataloader_pin_memory = True
cfg.train.auto_find_batch_size = True
cfg.train.tf32 = True
cfg.train.gradient_checkpointing = True

# --- Generation/eval config ---
cfg.generation.max_new_tokens = 120
cfg.generation.batch_size = cfg.train.per_device_eval_batch_size

# --- W&B ---
cfg.wandb.enabled = True
cfg.wandb.project = "abstract-evaluator-mistral7b-sft"
cfg.wandb.entity = None
cfg.wandb.tags = ["mistral7b", "lora", "sft", "modular", "l4-24gb", "no-quant"]

cfg.ensure_dirs()
cfg.as_dict()


In [ ]:
set_global_seed(cfg.seed)
configure_wandb_dir(str(cfg.wandb.dir))

log_dir = cfg.output_root / "logs"
logger = setup_logger(
    name=f"{cfg.run_name}_pipeline",
    log_dir=log_dir,
    log_file=f"{cfg.run_name}.log",
)
logger.info("Initialized run config: %s", json.dumps(cfg.as_dict(), ensure_ascii=False))
log_dir


In [ ]:
train_df, val_df, test_df = load_train_val_test_dfs(
    train_path=cfg.data_paths.train_path,
    val_path=cfg.data_paths.val_path,
    test_path=cfg.data_paths.test_path,
)

train_df, val_df, test_df = clean_train_val_test(train_df, val_df, test_df)

print("Shapes:", train_df.shape, val_df.shape, test_df.shape)
print("Train score dist:", score_distribution(train_df))
print("Val score dist:", score_distribution(val_df))
print("Test score dist:", score_distribution(test_df))


In [ ]:
train_df = add_messages_and_targets(train_df)
val_df = add_messages_and_targets(val_df)
test_df = add_messages_and_targets(test_df)

print(json.dumps(train_df.iloc[0]["messages"], indent=2, ensure_ascii=False)[:2000])


In [ ]:
jsonl_paths = export_split_jsonl(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    output_dir=cfg.output_root / "jsonl",
)

ds = to_hf_dataset_dict(train_df, val_df, test_df)

print("JSONL paths:", jsonl_paths)
print(ds)


In [ ]:
# Optional: token-length diagnostics (loads base tokenizer/model)
RUN_TOKEN_STATS = False

if RUN_TOKEN_STATS:
    lens = estimate_mistral_token_percentiles(
        model_name=cfg.model_name,
        max_seq_length=cfg.max_seq_length,
        df=pd.concat([train_df, val_df, test_df], ignore_index=True),
    )
    print(lens)


In [ ]:
# Main training run (fresh start flow; no resume-from-epoch-2 behavior)
RUN_TRAINING = True

output_dir = cfg.output_root / "models" / cfg.run_name

print({"train_rows": len(train_df), "val_rows": len(val_df), "test_rows": len(test_df)})

if RUN_TRAINING:
    run_info = train_mistral7(
        cfg=cfg,
        ds=ds,
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        include_bertscore_for_epoch_eval=False,
        run_epoch_generation_eval=True,
        run_epoch_test_eval=False,
        checkpoint_every_n_epochs=1,
        generation_eval_every_n_epochs=1,
        resume_from_checkpoint=None,
        logger=logger,
    )
else:
    print("RUN_TRAINING=False -> using eval-only mode with existing adapters.")
    run_info = {
        "model_name": cfg.model_name,
        "run_name": cfg.run_name,
        "output_dir": str(output_dir),
        "adapter_dir": str(output_dir / "best_adapter"),
        "epoch_adapter_dir": str(output_dir / "epoch_adapters"),
        "eval_dir": str(cfg.output_root / "eval" / cfg.run_name),
    }

run_info


In [ ]:
# Optional: evaluate base model (no finetuning) with BERTScore
RUN_BASE_EVAL = False

if RUN_BASE_EVAL:
    base_metrics = evaluate_base_model(
        cfg=cfg,
        val_df=val_df,
        test_df=test_df,
        include_bertscore=True,
        logger=logger,
    )

    display(pd.DataFrame([base_metrics]))


In [ ]:
# Optional: evaluate best adapter only (BERTScore included)
RUN_BEST_ADAPTER_EVAL = False

if RUN_BEST_ADAPTER_EVAL:
    best_adapter = Path(run_info["adapter_dir"])
    if not best_adapter.exists():
        raise FileNotFoundError(f"Best adapter not found: {best_adapter}")

    best_metrics = evaluate_single_adapter(
        cfg=cfg,
        adapter_dir=best_adapter,
        tag="best",
        val_df=val_df,
        test_df=test_df,
        include_bertscore=True,
        logger=logger,
    )

    display(pd.DataFrame([best_metrics]))


In [ ]:
# Separate generation step (model-agnostic utility)
# This demonstrates using generate_predictions_from_messages independently.
RUN_GENERATION_STEP_ONLY = False

if RUN_GENERATION_STEP_ONLY:
    from experiments.mistral.utils.chat import make_inference_prompt

    adapter_dir = Path(run_info["adapter_dir"])
    model, tokenizer = load_mistral_model_for_inference(
        model_name=cfg.model_name,
        adapter_dir=adapter_dir if adapter_dir.exists() else None,
        max_seq_length=cfg.max_seq_length,
    )

    demo_pred = generate_predictions_from_messages(
        eval_df=val_df.head(8),
        model=model,
        tokenizer=tokenizer,
        make_inference_prompt_fn=make_inference_prompt,
        parse_score_fn=parse_score,
        parse_rationale_fn=parse_rationale,
        max_seq_length=cfg.max_seq_length,
        max_new_tokens=cfg.generation.max_new_tokens,
        batch_size=2,
    )

    demo_pred.head()
